# Training a circuit on MNIST with data-driven initialization

This notebook shows an end-to-end example of **unsupervised density estimation** on MNIST. We construct a categorical probabilistic circuit, initialize its parameters from training images with `DataDrivenInitialization`, optimize it by maximum likelihood, and evaluate its negative log-likelihood (NLL) and bits per dimension (bpd).

The labels are deliberately ignored: the circuit learns a distribution over images, $p_\theta(\mathbf{x})$. For the theory behind population propagation, smoothing, and symmetry breaking, see the [data-driven initialization handbook](../docs/handbooks/data-driven-initialization.md).

## 1. Configuration

The default values below keep the tutorial reasonably small. For a full experiment, use all 60,000 training images, more circuit units, and more epochs. MNIST is downloaded to the user cache rather than inside the repository.

In [ ]:
import math
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torch.optim import Adam
from torch.utils.data import DataLoader, TensorDataset
from torchvision.datasets import MNIST

import cirkit.symbolic.functional as sf
from cirkit.pipeline import PipelineContext
from cirkit.templates.data_modalities import image_data
from cirkit.templates.initialization import DataDrivenInitialization

SEED = 42
IMAGE_SHAPE = (1, 28, 28)
NUM_PIXELS = 28 * 28

# Tutorial-sized experiment. Increase these values for a real training run.
TRAIN_SAMPLES = 10_000
VALID_SAMPLES = 2_000
TEST_SAMPLES = 2_000
INITIALIZATION_SAMPLES = 2_000
NUM_UNITS = 8
NUM_EPOCHS = 3
BATCH_SIZE = 256
LEARNING_RATE = 1e-2

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## 2. Load discrete MNIST images

Categorical image inputs expect integer observations in $\{0,\ldots,255\}$. The `data` tensor stored by `torchvision.datasets.MNIST` already has dtype `uint8`, so no floating-point normalization is needed. We flatten each image to the `(batch, 784)` representation expected by the compiled circuit.

In [ ]:
data_root = Path.home() / ".cache" / "cirkit" / "datasets"
mnist_train = MNIST(root=data_root, train=True, download=True)
mnist_test = MNIST(root=data_root, train=False, download=True)

generator = torch.Generator().manual_seed(SEED)
indices = torch.randperm(len(mnist_train), generator=generator)
train_indices = indices[:TRAIN_SAMPLES]
valid_indices = indices[TRAIN_SAMPLES : TRAIN_SAMPLES + VALID_SAMPLES]

train_images = mnist_train.data[train_indices].reshape(-1, NUM_PIXELS).contiguous()
valid_images = mnist_train.data[valid_indices].reshape(-1, NUM_PIXELS).contiguous()
test_images = mnist_test.data[:TEST_SAMPLES].reshape(-1, NUM_PIXELS).contiguous()

train_loader = DataLoader(
    TensorDataset(train_images), batch_size=BATCH_SIZE, shuffle=True
)
valid_loader = DataLoader(
    TensorDataset(valid_images), batch_size=BATCH_SIZE, shuffle=False
)
test_loader = DataLoader(
    TensorDataset(test_images), batch_size=BATCH_SIZE, shuffle=False
)

train_images.shape, train_images.dtype, (train_images.min().item(), train_images.max().item())

In [ ]:
figure, axes = plt.subplots(1, 8, figsize=(12, 2))
for axis, image in zip(axes, train_images[:8]):
    axis.imshow(image.reshape(28, 28), cmap="gray", vmin=0, vmax=255)
    axis.axis("off")
figure.suptitle("MNIST training observations")
plt.tight_layout()

## 3. Construct and initialize the symbolic circuit

`image_data` first constructs a fixed smooth and decomposable circuit from a QuadTree region graph. Because both `data` and `initialization` are supplied, Cirkit then propagates populations through that structure and initializes:

- categorical input probabilities from smoothed pixel counts;
- sum weights from the sizes of data clusters; and
- replicated tensor units with seeded symmetry-breaking noise.

Only `INITIALIZATION_SAMPLES` images are needed for this starting point; the optimizer will still train on all `TRAIN_SAMPLES` images. Here `alpha=0.5` is a pseudocount added to every pixel category and every mixture branch.

In [ ]:
initialization_data = train_images[:INITIALIZATION_SAMPLES].to(device)

symbolic_circuit = image_data(
    IMAGE_SHAPE,
    region_graph="quad-tree-2",
    input_layer="categorical",
    num_input_units=NUM_UNITS,
    sum_product_layer="cp",
    num_sum_units=NUM_UNITS,
    use_mixing_weights=True,
    input_sharing="none",
    data=initialization_data,
    initialization=DataDrivenInitialization(
        alpha=0.5,
        noise_scale=0.1,
        seed=SEED,
    ),
)

assert symbolic_circuit.is_smooth
assert symbolic_circuit.is_decomposable
len(list(symbolic_circuit.layers))

## 4. Compile the circuit

The `lse-sum` semiring evaluates the circuit in log space for numerical stability. Both categorical probabilities and sum weights use softmax parameterizations, so this smooth and decomposable circuit is normalized. Its log-partition function should therefore be approximately zero.

In [ ]:
context = PipelineContext(
    backend="torch", semiring="lse-sum", fold=True, optimize=True
)
circuit = context.compile(symbolic_circuit).to(device)
partition = context.compile(sf.integrate(symbolic_circuit)).to(device)

with torch.no_grad():
    log_partition = partition().item()

num_parameters = sum(parameter.numel() for parameter in circuit.parameters())
print(f"Device: {device}")
print(f"Trainable parameters: {num_parameters:,}")
print(f"log Z: {log_partition:.6f}")

## 5. Evaluate the initialization

Before applying any gradient update, we measure the validation NLL. Bits per dimension divides the NLL by $784\log 2$, making the value easier to compare across image models.

In [ ]:
@torch.inference_mode()
def evaluate(model, data_loader):
    model.eval()
    total_nll = 0.0
    total_samples = 0

    for (batch,) in data_loader:
        batch = batch.to(device=device, dtype=torch.long)
        log_likelihood = model(batch).flatten()
        total_nll += (-log_likelihood).sum().item()
        total_samples += batch.size(0)

    mean_nll = total_nll / total_samples
    bits_per_dimension = mean_nll / (NUM_PIXELS * math.log(2.0))
    return mean_nll, bits_per_dimension

In [ ]:
initial_val_nll, initial_val_bpd = evaluate(circuit, valid_loader)
print(f"Before training - validation NLL: {initial_val_nll:.3f}")
print(f"Before training - validation bpd: {initial_val_bpd:.3f}")

## 6. Maximum-likelihood training

`DataDrivenInitialization` chooses the starting parameters; it does not replace training. Since the circuit is normalized, maximizing log-likelihood is simply minimizing `-circuit(batch).mean()`. All parameters remain ordinary PyTorch parameters and can be optimized with Adam.

In [ ]:
optimizer = Adam(circuit.parameters(), lr=LEARNING_RATE)
history = []

for epoch in range(1, NUM_EPOCHS + 1):
    circuit.train()
    train_nll_sum = 0.0
    train_samples = 0

    for (batch,) in train_loader:
        batch = batch.to(device=device, dtype=torch.long)
        optimizer.zero_grad(set_to_none=True)

        loss = -circuit(batch).mean()
        loss.backward()
        optimizer.step()

        train_nll_sum += loss.detach().item() * batch.size(0)
        train_samples += batch.size(0)

    train_nll = train_nll_sum / train_samples
    train_bpd = train_nll / (NUM_PIXELS * math.log(2.0))
    val_nll, val_bpd = evaluate(circuit, valid_loader)
    history.append(
        {
            "epoch": epoch,
            "train_nll": train_nll,
            "train_bpd": train_bpd,
            "val_nll": val_nll,
            "val_bpd": val_bpd,
        }
    )
    print(
        f"Epoch {epoch:02d} | "
        f"train NLL {train_nll:.3f}, bpd {train_bpd:.3f} | "
        f"validation NLL {val_nll:.3f}, bpd {val_bpd:.3f}"
    )

In [ ]:
epochs = [entry["epoch"] for entry in history]
train_bpd = [entry["train_bpd"] for entry in history]
validation_bpd = [entry["val_bpd"] for entry in history]

plt.figure(figsize=(6, 4))
plt.plot(epochs, train_bpd, marker="o", label="train")
plt.plot(epochs, validation_bpd, marker="o", label="validation")
plt.xlabel("Epoch")
plt.ylabel("Bits per dimension")
plt.title("MNIST density-estimation training")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

## 7. Test-set evaluation

The final cell evaluates the trained circuit on held-out MNIST images. This tutorial configuration is intentionally small, so the result is a functional demonstration rather than a competitive benchmark.

In [ ]:
test_nll, test_bpd = evaluate(circuit, test_loader)
print(f"Test NLL: {test_nll:.3f}")
print(f"Test bits per dimension: {test_bpd:.3f}")
print(f"Validation bpd before training: {initial_val_bpd:.3f}")
print(f"Validation bpd after training:  {history[-1]['val_bpd']:.3f}")